# 11 — GPU Scaling Study

Measures **speedup as the number of GPUs varies (1 → 2 → 4 → 8)** for:

| Experiment | What scales | Metric |
|---|---|---|
| **A. WJ Reranking** | Query batch distributed across N GPUs (corpus pre-loaded) | Reranking QPS |
| **B. MLP Training** | DataParallel batch across N GPUs | Training throughput (samples/s) |

**Hardware:** DGX — 8× A100-SXM4-80 GB  
**Dataset:** Full 233k OpenStreetMap park polygons (187,019 corpus + 46,754 queries)  
**Fixed:** K=1000 candidates

In [1]:
import gc
import os
import pickle
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# Save plots next to this notebook
NOTEBOOK_DIR = Path('.').resolve()
print(f"Notebook dir (plots will be saved here): {NOTEBOOK_DIR}")

# ── Discover available GPUs ───────────────────────────────────────────────────
N_AVAILABLE = torch.cuda.device_count()
print(f"Available CUDA devices: {N_AVAILABLE}")
for i in range(N_AVAILABLE):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {props.name}  {props.total_memory / 1024**3:.1f} GB")

GPU_COUNTS = [n for n in [1, 2, 4, 8] if n <= N_AVAILABLE]
print(f"\nWill benchmark GPU counts: {GPU_COUNTS}")

QUERY_START_FULL = 187019
CANDIDATE_K      = 1000
TRAIN_BATCH_GPU  = 256
WARMUP_QUERIES   = 200
N_TIMING_QUERIES = 2000
TRAIN_STEPS      = 50

Notebook dir (plots will be saved here): /raid/ruban/hpmlproj/term_project
Available CUDA devices: 8
  cuda:0  NVIDIA A100-SXM4-80GB  79.2 GB
  cuda:1  NVIDIA A100-SXM4-80GB  79.2 GB
  cuda:2  NVIDIA A100-SXM4-80GB  79.2 GB
  cuda:3  NVIDIA A100-SXM4-80GB  79.2 GB
  cuda:4  NVIDIA A100-SXM4-80GB  79.2 GB
  cuda:5  NVIDIA A100-SXM4-80GB  79.2 GB
  cuda:6  NVIDIA A100-SXM4-80GB  79.2 GB
  cuda:7  NVIDIA A100-SXM4-80GB  79.2 GB

Will benchmark GPU counts: [1, 2, 4, 8]


## Model & Data

In [2]:
class QuadtreeCompressorV1Fixed(nn.Module):
    """MLP Compressor: 18220 → 4096 → 1024 → 512, log1p preprocessing."""
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096,  1024, bias=False),  nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024,  out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        x = torch.log1p(x * 1e6)
        return self.net(x)


print("Loading quadtree vectors ...")
qt          = np.load("/tmp/qtree_vectors_full.npy")
corpus_qt   = qt[:QUERY_START_FULL].astype(np.float32)
query_qt    = qt[QUERY_START_FULL:].astype(np.float32)
corpus_sums = corpus_qt.sum(axis=1).astype(np.float32)
DIM         = qt.shape[1]
print(f"Corpus: {corpus_qt.shape}  Query: {query_qt.shape}  dim={DIM}")

Loading quadtree vectors ...
Corpus: (187019, 18220)  Query: (46754, 18220)  dim=18220


## Experiment A — WJ Reranking Scaling

**Strategy**: Pre-load the full corpus onto each GPU once (setup cost, not timed).  
During the timed region, only small query shards are transferred H→D.  
Query batch is split evenly across N GPUs via `ThreadPoolExecutor`.

**Communication pattern per GPU (timed region only):**
- H→D: query shard (Q/N × inner_batch × D × 4B) — ~MB scale
- D→H: sorted candidate indices (Q/N × K × 8B)
- Sync: `torch.cuda.synchronize()` after each inner batch

**Parallel time complexity:** T(N) = O(Q × K × D / N) — perfectly data-parallel  
**Read/write contention:** None — each GPU owns its corpus copy and a disjoint query shard

In [3]:
# Pre-load corpus onto every GPU once — this is setup, not included in QPS timing
print("Pre-loading corpus onto each GPU ...")
corpus_tensors = {}
csums_tensors  = {}
for g in range(max(GPU_COUNTS)):
    dev = torch.device(f'cuda:{g}')
    corpus_tensors[g] = torch.from_numpy(corpus_qt).to(dev)
    csums_tensors[g]  = torch.from_numpy(corpus_sums).to(dev)
    torch.cuda.synchronize(g)
    print(f"  cuda:{g} — {corpus_tensors[g].nbytes / 1024**3:.1f} GB loaded")


def rerank_shard_preloaded(query_shard, candidate_ids, gpu_id, inner_batch=16):
    """
    Exact WJ reranking on a single GPU using pre-loaded corpus.
    Processes queries in inner_batch chunks to stay within GPU memory.

    Memory per inner batch: inner_batch × K × D × 4B
      = 16 × 1000 × 18220 × 4B ≈ 1.1 GB  (safe on 80 GB A100)
    """
    dev      = torch.device(f'cuda:{gpu_id}')
    corpus_t = corpus_tensors[gpu_id]   # already resident on GPU
    csums_t  = csums_tensors[gpu_id]
    results  = []

    for start in range(0, len(query_shard), inner_batch):
        end    = min(start + inner_batch, len(query_shard))
        q_t    = torch.from_numpy(query_shard[start:end]).to(dev)    # H→D (small)
        ids_t  = torch.from_numpy(candidate_ids[start:end]).to(dev)  # H→D (small)

        cand_t = corpus_t[ids_t]                                       # (B, K, D)
        mins   = torch.minimum(q_t[:, None, :], cand_t).sum(dim=2)    # (B, K)
        q_sums = q_t.sum(dim=1, keepdim=True)
        maxs   = q_sums + csums_t[ids_t] - mins
        wj     = mins / maxs.clamp_min(1e-10)

        order   = torch.argsort(wj, dim=1, descending=True).cpu().numpy()  # D→H
        ids_cpu = candidate_ids[start:end]
        for i in range(end - start):
            results.append(ids_cpu[i][order[i]])

        del q_t, ids_t, cand_t, mins, maxs, wj, order

    torch.cuda.synchronize(gpu_id)
    return results


def benchmark_reranking(n_gpus, candidate_k=CANDIDATE_K,
                         n_queries=N_TIMING_QUERIES, warmup=WARMUP_QUERIES):
    print(f"\n--- Reranking: n_gpus={n_gpus}, K={candidate_k}, "
          f"n_queries={n_queries} ---")

    rng = np.random.default_rng(42)
    all_cands = rng.integers(0, len(corpus_qt),
                             size=(n_queries + warmup, candidate_k),
                             dtype=np.int64)
    q_all = query_qt[:n_queries + warmup]

    def run_shard(sq, si, g):
        return rerank_shard_preloaded(sq, si, g)

    # Warmup
    sz = max(1, warmup // n_gpus)
    with ThreadPoolExecutor(max_workers=n_gpus) as ex:
        fs = [ex.submit(run_shard,
                        q_all[g*sz : min((g+1)*sz, warmup)],
                        all_cands[g*sz : min((g+1)*sz, warmup)], g)
              for g in range(n_gpus) if g*sz < warmup]
        [f.result() for f in fs]
    for g in range(n_gpus):
        torch.cuda.synchronize(g)

    # Timed
    tq   = q_all[warmup : warmup + n_queries]
    ti   = all_cands[warmup : warmup + n_queries]
    sz   = max(1, n_queries // n_gpus)

    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n_gpus) as ex:
        fs = [ex.submit(run_shard,
                        tq[g*sz : min((g+1)*sz, n_queries)],
                        ti[g*sz : min((g+1)*sz, n_queries)], g)
              for g in range(n_gpus) if g*sz < n_queries]
        [f.result() for f in fs]
    for g in range(n_gpus):
        torch.cuda.synchronize(g)
    total_s = time.perf_counter() - t0

    qps = n_queries / total_s
    print(f"  total_s={total_s:.2f}s  QPS={qps:.1f}")
    return {'n_gpus': n_gpus, 'qps': qps, 'total_s': total_s,
            'n_queries': n_queries, 'candidate_k': candidate_k}


rerank_results = []
for n in GPU_COUNTS:
    r = benchmark_reranking(n)
    rerank_results.append(r)
    gc.collect()

base_rqps = rerank_results[0]['qps']
for r in rerank_results:
    r['speedup'] = r['qps'] / base_rqps

print("\nReranking scaling summary:")
print(f"{'GPUs':>6} {'QPS':>10} {'Speedup':>10} {'Efficiency':>12}")
for r in rerank_results:
    eff = r['speedup'] / r['n_gpus'] * 100
    print(f"{r['n_gpus']:>6} {r['qps']:>10.1f} {r['speedup']:>10.2f}x {eff:>11.1f}%")

Pre-loading corpus onto each GPU ...
  cuda:0 — 12.7 GB loaded
  cuda:1 — 12.7 GB loaded
  cuda:2 — 12.7 GB loaded
  cuda:3 — 12.7 GB loaded
  cuda:4 — 12.7 GB loaded
  cuda:5 — 12.7 GB loaded
  cuda:6 — 12.7 GB loaded
  cuda:7 — 12.7 GB loaded

--- Reranking: n_gpus=1, K=1000, n_queries=2000 ---
  total_s=0.67s  QPS=2972.2

--- Reranking: n_gpus=2, K=1000, n_queries=2000 ---
  total_s=0.34s  QPS=5836.0

--- Reranking: n_gpus=4, K=1000, n_queries=2000 ---
  total_s=0.19s  QPS=10711.2

--- Reranking: n_gpus=8, K=1000, n_queries=2000 ---
  total_s=0.10s  QPS=20677.3

Reranking scaling summary:
  GPUs        QPS    Speedup   Efficiency
     1     2972.2       1.00x       100.0%
     2     5836.0       1.96x        98.2%
     4    10711.2       3.60x        90.1%
     8    20677.3       6.96x        87.0%


## Experiment B — MLP Training Throughput Scaling

**Strategy**: `torch.nn.DataParallel` across N GPUs.  
Total batch = N × 256. Measure samples/second over 50 timed steps.

**Communication pattern (DataParallel):**
- Forward: scatter mini-batch to N GPUs
- Backward: NCCL all-reduce gradients over NVLink (~600 GB/s on DGX A100)
- Sync point: implicit at `loss.backward()` completion

**Synchronization overhead:** O(P) all-reduce per step, P=79.4M params (~318 MB)  
**Parallel time complexity:** T(N) = O(P × B/N) compute + O(P) communication

In [4]:
def benchmark_training(n_gpus, dim=18220, out_dim=512,
                        batch_per_gpu=TRAIN_BATCH_GPU, n_steps=TRAIN_STEPS):
    print(f"\n--- Training: n_gpus={n_gpus}, batch/gpu={batch_per_gpu}, "
          f"steps={n_steps} ---")

    device_ids = list(range(n_gpus))
    primary    = torch.device(f'cuda:{device_ids[0]}')
    total_batch = batch_per_gpu * n_gpus

    model = QuadtreeCompressorV1Fixed(dim, out_dim).to(primary)
    if n_gpus > 1:
        model = nn.DataParallel(model, device_ids=device_ids)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    # Synthetic sparse data mimicking quadtree vectors (~73% sparsity, L1-normalised)
    rng  = np.random.default_rng(0)
    data = rng.exponential(1e-9, size=(total_batch * (n_steps + 10), dim)).astype(np.float32)
    mask = rng.random(size=data.shape) < 0.73
    data[mask] = 0.0
    data /= data.sum(axis=1, keepdims=True).clip(min=1e-12)
    loader    = DataLoader(TensorDataset(torch.from_numpy(data)),
                           batch_size=total_batch, shuffle=False,
                           num_workers=0, pin_memory=True)
    loader_it = iter(loader)

    def step():
        nonlocal loader_it
        try:
            (xb,) = next(loader_it)
        except StopIteration:
            loader_it = iter(loader)
            (xb,) = next(loader_it)
        xb  = xb.to(primary)
        out = model(xb)
        loss = -F.normalize(out, dim=1).var(dim=0).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        return xb.size(0)

    # Warmup
    print("  Warmup (5 steps) ...")
    for _ in range(5):
        step()
    torch.cuda.synchronize(device_ids[0])

    # Timed
    print(f"  Timing {n_steps} steps ...")
    total_samples = 0
    t0 = time.perf_counter()
    for _ in range(n_steps):
        total_samples += step()
    torch.cuda.synchronize(device_ids[0])
    total_s = time.perf_counter() - t0

    sps = total_samples / total_s
    print(f"  total_s={total_s:.2f}s  samples/s={sps:.1f}  (total_batch={total_batch})")

    del model, optimizer, loader
    torch.cuda.empty_cache()
    gc.collect()
    return {'n_gpus': n_gpus, 'samples_per_sec': sps, 'total_s': total_s,
            'total_samples': total_samples, 'batch_total': total_batch}


train_results = []
for n in GPU_COUNTS:
    r = benchmark_training(n, dim=DIM)
    train_results.append(r)

base_sps = train_results[0]['samples_per_sec']
for r in train_results:
    r['speedup'] = r['samples_per_sec'] / base_sps

print("\nTraining scaling summary:")
print(f"{'GPUs':>6} {'Samples/s':>12} {'Speedup':>10} {'Efficiency':>12}")
for r in train_results:
    eff = r['speedup'] / r['n_gpus'] * 100
    print(f"{r['n_gpus']:>6} {r['samples_per_sec']:>12.1f} "
          f"{r['speedup']:>10.2f}x {eff:>11.1f}%")


--- Training: n_gpus=1, batch/gpu=256, steps=50 ---
  Warmup (5 steps) ...
  Timing 50 steps ...
  total_s=0.54s  samples/s=23545.5  (total_batch=256)

--- Training: n_gpus=2, batch/gpu=256, steps=50 ---
  Warmup (5 steps) ...
  Timing 50 steps ...
  total_s=1.49s  samples/s=17141.5  (total_batch=512)

--- Training: n_gpus=4, batch/gpu=256, steps=50 ---
  Warmup (5 steps) ...
  Timing 50 steps ...
  total_s=2.63s  samples/s=19450.1  (total_batch=1024)

--- Training: n_gpus=8, batch/gpu=256, steps=50 ---
  Warmup (5 steps) ...
  Timing 50 steps ...
  total_s=5.12s  samples/s=20008.8  (total_batch=2048)

Training scaling summary:
  GPUs    Samples/s    Speedup   Efficiency
     1      23545.5       1.00x       100.0%
     2      17141.5       0.73x        36.4%
     4      19450.1       0.83x        20.7%
     8      20008.8       0.85x        10.6%


## Save Results

In [5]:
scaling_results = {
    'reranking':  rerank_results,
    'training':   train_results,
    'gpu_counts': GPU_COUNTS,
}
with open('/tmp/results_gpu_scaling.pkl', 'wb') as f:
    pickle.dump(scaling_results, f)
print('Saved to /tmp/results_gpu_scaling.pkl')

Saved to /tmp/results_gpu_scaling.pkl


## Speedup Plots

In [6]:
with open('/tmp/results_gpu_scaling.pkl', 'rb') as f:
    res = pickle.load(f)

rr   = res['reranking']
tr   = res['training']
gpus = [r['n_gpus'] for r in rr]

rerank_qps     = [r['qps']             for r in rr]
rerank_speedup = [r['speedup']         for r in rr]
rerank_eff     = [r['speedup'] / r['n_gpus'] * 100 for r in rr]

train_sps      = [r['samples_per_sec'] for r in tr]
train_speedup  = [r['speedup']         for r in tr]
train_eff      = [r['speedup'] / r['n_gpus'] * 100 for r in tr]

ideal = gpus

COLORS = {'rerank': '#1f77b4', 'train': '#ff7f0e', 'ideal': '#aaaaaa'}

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle(
    'GPU Scaling Study — WJ Reranking & MLP Training\n'
    'DGX 8× A100-SXM4-80GB · Full 233k OSM Park Polygons · K=1000',
    fontsize=13, fontweight='bold'
)

# (a) Reranking QPS
ax = axes[0, 0]
ax.plot(gpus, rerank_qps, 'o-', color=COLORS['rerank'], lw=2, ms=8)
for x, y in zip(gpus, rerank_qps):
    ax.annotate(f'{y:,.0f}', (x, y), textcoords='offset points',
                xytext=(0, 9), ha='center', fontsize=9)
ax.set_xlabel('Number of GPUs'); ax.set_ylabel('Reranking QPS')
ax.set_title('(a) WJ Reranking Throughput vs. GPUs')
ax.set_xticks(gpus)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(True, alpha=0.3)

# (b) Training throughput
ax = axes[0, 1]
ax.plot(gpus, train_sps, 's-', color=COLORS['train'], lw=2, ms=8)
for x, y in zip(gpus, train_sps):
    ax.annotate(f'{y:,.0f}', (x, y), textcoords='offset points',
                xytext=(0, 9), ha='center', fontsize=9)
ax.set_xlabel('Number of GPUs'); ax.set_ylabel('Training Throughput (samples/s)')
ax.set_title('(b) MLP Training Throughput vs. GPUs')
ax.set_xticks(gpus)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(True, alpha=0.3)

# (c) Speedup
ax = axes[1, 0]
ax.plot(gpus, ideal,          '--', color=COLORS['ideal'],  lw=1.5, label='Ideal (linear)')
ax.plot(gpus, rerank_speedup, 'o-', color=COLORS['rerank'], lw=2, ms=8, label='WJ Reranking')
ax.plot(gpus, train_speedup,  's-', color=COLORS['train'],  lw=2, ms=8, label='MLP Training')
for x, y in zip(gpus, rerank_speedup):
    ax.annotate(f'{y:.2f}×', (x, y), textcoords='offset points',
                xytext=(-14, 6), fontsize=8, color=COLORS['rerank'])
for x, y in zip(gpus, train_speedup):
    ax.annotate(f'{y:.2f}×', (x, y), textcoords='offset points',
                xytext=(6, -14), fontsize=8, color=COLORS['train'])
ax.set_xlabel('Number of GPUs'); ax.set_ylabel('Speedup (relative to 1 GPU)')
ax.set_title('(c) Speedup vs. Number of GPUs')
ax.set_xticks(gpus); ax.grid(True, alpha=0.3); ax.legend()

# (d) Parallel efficiency
ax = axes[1, 1]
ax.axhline(100, linestyle='--', color=COLORS['ideal'], lw=1.5, label='Ideal (100%)')
ax.plot(gpus, rerank_eff, 'o-', color=COLORS['rerank'], lw=2, ms=8, label='WJ Reranking')
ax.plot(gpus, train_eff,  's-', color=COLORS['train'],  lw=2, ms=8, label='MLP Training')
for x, y in zip(gpus, rerank_eff):
    ax.annotate(f'{y:.1f}%', (x, y), textcoords='offset points',
                xytext=(-16, 6), fontsize=8, color=COLORS['rerank'])
for x, y in zip(gpus, train_eff):
    ax.annotate(f'{y:.1f}%', (x, y), textcoords='offset points',
                xytext=(6, -14), fontsize=8, color=COLORS['train'])
ax.set_xlabel('Number of GPUs'); ax.set_ylabel('Parallel Efficiency (%)')
ax.set_title('(d) Parallel Efficiency vs. Number of GPUs')
ax.set_xticks(gpus); ax.set_ylim(0, 120); ax.grid(True, alpha=0.3); ax.legend()

plt.tight_layout(rect=[0, 0, 1, 0.95])

plot_path = NOTEBOOK_DIR / 'speedup_plot.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to {plot_path}')

Plot saved to /raid/ruban/hpmlproj/term_project/speedup_plot.png


## Final Table

In [7]:
print("\n" + "=" * 78)
print("GPU SCALING RESULTS")
print("=" * 78)
print(f"{'GPUs':>5}  {'Rerank QPS':>12}  {'Rerank Speedup':>15}  "
      f"{'Train Samp/s':>13}  {'Train Speedup':>13}")
print("-" * 78)
for rr_, tr_ in zip(rerank_results, train_results):
    print(f"{rr_['n_gpus']:>5}  {rr_['qps']:>12,.1f}  "
          f"{rr_['speedup']:>14.2f}x  "
          f"{tr_['samples_per_sec']:>13,.1f}  "
          f"{tr_['speedup']:>12.2f}x")
print("=" * 78)
print(f"Reranking: K={CANDIDATE_K}, {N_TIMING_QUERIES} timed queries, full 233k corpus, "
      f"corpus pre-loaded to each GPU")
print(f"Training:  {TRAIN_STEPS} steps × {TRAIN_BATCH_GPU} samples/GPU, "
      f"AdamW, DataParallel, MLP 79.4M params")
print(f"\nPlot saved to: {NOTEBOOK_DIR / 'speedup_plot.png'}")


GPU SCALING RESULTS
 GPUs    Rerank QPS   Rerank Speedup   Train Samp/s  Train Speedup
------------------------------------------------------------------------------
    1       2,972.2            1.00x       23,545.5          1.00x
    2       5,836.0            1.96x       17,141.5          0.73x
    4      10,711.2            3.60x       19,450.1          0.83x
    8      20,677.3            6.96x       20,008.8          0.85x
Reranking: K=1000, 2000 timed queries, full 233k corpus, corpus pre-loaded to each GPU
Training:  50 steps × 256 samples/GPU, AdamW, DataParallel, MLP 79.4M params

Plot saved to: /raid/ruban/hpmlproj/term_project/speedup_plot.png
